In [1]:
import numpy as np
import glob
import os
import joblib
import tensorflow as tf
from pyscf import gto, scf
from tensorflow.keras import layers, models, callbacks, optimizers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =============================================================================
# 0. CONFIGURATION & DIRECTORY SETUP
# =============================================================================
N_ATOMS = 70 
SYSTEM_NAME = f"H{N_ATOMS}"
CHECKPOINT_PATTERN = f"data_checkpoint_h{N_ATOMS}_run*_rank*_step*.npz" 

DEPLOY_DIR = "deployment_objects"
os.makedirs(DEPLOY_DIR, exist_ok=True)
MODEL_SAVE_NAME  = os.path.join(DEPLOY_DIR, f"NN_{SYSTEM_NAME}_DeltaHF.keras")

print(f">>> STARTING DELTA-LEARNING PIPELINE FOR: {SYSTEM_NAME}")

# =============================================================================
# 1. PHYSICS SETUP & REFERENCE GENERATION
# =============================================================================
print(f"\n>>> 1. Generating Physics Constants & Baseline (HF)...")

mol = gto.M(
    atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], 
    basis="sto-6g", verbose=0, spin=0
)
mf = scf.UHF(mol)
mf.kernel()

S = mf.get_ovlp()
eigvals, eigvecs = np.linalg.eigh(S)
S_sqrt = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
S_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T

h_core_ao = mf.get_hcore()
h_core_lowdin = S_inv_sqrt @ h_core_ao @ S_inv_sqrt

# Exact HF Baseline
P_hf_spin = mf.make_rdm1()
P_hf_ao = P_hf_spin[0] + P_hf_spin[1] if P_hf_spin.ndim == 3 else P_hf_spin
# Contravariant transformation for Density Matrix
P_hf_lowdin = S_sqrt @ P_hf_ao @ S_sqrt
E_hf = mf.e_tot

C_sim = mf.mo_coeff[0] 

np.save(os.path.join(DEPLOY_DIR, "S_sqrt.npy"), S_sqrt)
np.save(os.path.join(DEPLOY_DIR, "C_sim.npy"), C_sim)
np.save(os.path.join(DEPLOY_DIR, "h_core_lowdin.npy"), h_core_lowdin)
np.save(os.path.join(DEPLOY_DIR, "P_hf_lowdin.npy"), P_hf_lowdin)
np.save(os.path.join(DEPLOY_DIR, "E_hf.npy"), np.array([E_hf]))

# =============================================================================
# 2. BULLETPROOF CUSTOM LOSS
# =============================================================================
@tf.keras.utils.register_keras_serializable()
def log_cosh_loss(y_true, y_pred):
    """Robust loss with strict shape enforcement to prevent broadcasting bugs."""
    y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)), dtype=y_pred.dtype)
    return tf.reduce_mean(tf.math.log(tf.math.cosh(y_pred - y_true)))

# =============================================================================
# 3. LOAD & PROCESS WALKER DATA
# =============================================================================
print(f"\n>>> 3. Loading Walker Checkpoints...")
files = glob.glob(CHECKPOINT_PATTERN)
if not files: raise ValueError("No files found!")

all_GA, all_GB, all_E = [], [], []
nbasis = h_core_ao.shape[0]

for f in sorted(files):
    data = np.load(f)
    all_GA.append(data['GA'])
    all_GB.append(data['GB'])
    all_E.append(data['E'])

GA_raw = np.concatenate(all_GA, axis=0).reshape(-1, nbasis, nbasis)
GB_raw = np.concatenate(all_GB, axis=0).reshape(-1, nbasis, nbasis)
E_raw = np.concatenate(all_E, axis=0).real.reshape(-1)

print("    Transforming Walkers AO -> Lowdin...")
GA_lowdin = np.einsum('ij, bjk, kl -> bil', S_sqrt, GA_raw, S_sqrt)
GB_lowdin = np.einsum('ij, bjk, kl -> bil', S_sqrt, GB_raw, S_sqrt)

# =============================================================================
# 4. PREPARE TRAINING DATA (FIXED SHAPES)
# =============================================================================
print("\n>>> 4. Preparing Physics-Informed Delta Features...")

delta_E_raw = E_raw - E_hf
P_total = GA_lowdin + GB_lowdin
delta_P = P_total - P_hf_lowdin

# Exact 1-Body Delta Energy
E_1B_delta = np.einsum('ij, bji -> b', h_core_lowdin, delta_P).real

# Pure Correlation Residual
y_corr_raw = delta_E_raw - E_1B_delta

median_e = np.median(y_corr_raw)
mad_e = np.median(np.abs(y_corr_raw - median_e))
mask = (y_corr_raw > (median_e - 4 * mad_e)) & (y_corr_raw < (median_e + 4 * mad_e))

delta_P_clean = delta_P[mask]
y_corr_clean  = y_corr_raw[mask]

X_final = np.stack([np.real(delta_P_clean), np.imag(delta_P_clean)], axis=-1)
X_flipped = np.flip(X_final, axis=(1, 2))
y_flipped = y_corr_clean

X_augmented = np.concatenate([X_final, X_flipped], axis=0)
y_augmented = np.concatenate([y_corr_clean, y_flipped], axis=0)

X_train, X_test, y_train, y_test = train_test_split(X_augmented, y_augmented, test_size=0.2, random_state=42)

# STANDARDIZATION (Keeping targets strictly 2D for Keras)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_test_scaled  = X_scaler.transform(X_test_flat).reshape(X_test.shape)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)) # No flatten()
y_test_scaled  = y_scaler.transform(y_test.reshape(-1, 1))      # No flatten()

joblib.dump(X_scaler, os.path.join(DEPLOY_DIR, "X_scaler.save"))
joblib.dump(y_scaler, os.path.join(DEPLOY_DIR, "y_scaler.save"))

# =============================================================================
# 5. BUILD MLP BOTTLENECK MODEL
# =============================================================================
print("\n>>> 5. Building Regularized Correlation Model...")

def create_correlation_model(input_shape):
    p_in = layers.Input(shape=input_shape)
    x = layers.Flatten()(p_in)
    
    reg = regularizers.l2(1e-4)
    x = layers.Dense(256, activation='swish', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='swish', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(64, activation='swish', kernel_regularizer=reg)(x)
    
    e_corr_scaled = layers.Dense(1, name="Scaled_Corr_Prediction")(x)
    return models.Model(inputs=p_in, outputs=e_corr_scaled)

model = create_correlation_model(input_shape=(X_train.shape[1], X_train.shape[2], 2))
model.compile(loss=log_cosh_loss, optimizer=optimizers.Adamax(learning_rate=1e-3), metrics=['mae'])

history = model.fit(
    X_train_scaled, y_train_scaled,
    validation_data=(X_test_scaled, y_test_scaled),
    epochs=500, batch_size=128, verbose=1,
    callbacks=[
        callbacks.EarlyStopping(patience=40, restore_best_weights=True, monitor='val_loss'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=15, verbose=1)
    ]
)

# =============================================================================
# 6. EVALUATION
# =============================================================================
print("\n>>> 6. Final Evaluation...")
preds_scaled = model.predict(X_test_scaled)

preds_corr = y_scaler.inverse_transform(preds_scaled).flatten()
y_test_corr = y_scaler.inverse_transform(y_test_scaled).flatten()

mae = np.mean(np.abs(preds_corr - y_test_corr)) * 1000 
print(f"    Test Correlation MAE: {mae:.4f} mHa")

model.save(MODEL_SAVE_NAME)
print(f"    Model saved to {MODEL_SAVE_NAME}")

2026-02-28 17:13:02.646107: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


>>> STARTING DELTA-LEARNING PIPELINE FOR: H70

>>> 1. Generating Physics Constants & Baseline (HF)...

>>> 3. Loading Walker Checkpoints...
    Transforming Walkers AO -> Lowdin...

>>> 4. Preparing Physics-Informed Delta Features...

>>> 5. Building Regularized Correlation Model...


I0000 00:00:1772322746.901539 1409900 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46714 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


Epoch 1/500


2026-02-28 18:52:30.456748: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fd7000060c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-28 18:52:30.456777: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-02-28 18:52:30.529035: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-28 18:52:30.691452: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-02-28 18:52:30.855725: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-28 18:52:30.855757: I external

...

394/394 ━━━━━━━━━━━━━━━━━━━━ 43s 64ms/step - loss: 0.3479 - mae: 0.6335 - val_loss: 0.1592 - val_mae: 0.3252 - learning_rate: 0.0010
Epoch 2/500
394/394 ━━━━━━━━━━━━━━━━━━━━ 42s 5ms/step - loss: 0.1925 - mae: 0.3925 - val_loss: 0.1340 - val_mae: 0.2706 - learning_rate: 0.0010
Epoch 3/500
394/394 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1633 - mae: 0.3388 - val_loss: 0.1226 - val_mae: 0.2479 - learning_rate: 0.0010
Epoch 4/500
394/394 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1459 - mae: 0.3088 - val_loss: 0.1132 - val_mae: 0.2293 - learning_rate: 0.0010
Epoch 5/500
394/394 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.1328 - mae: 0.2891 - val_loss: 0.1051 - val_mae: 0.2198 - learning_rate: 0.0010
Epoch 6/500
394/394 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1222 - mae: 0.2734 - val_loss: 0.0988 - val_mae: 0.2176 - learning_rate: 0.0010
Epoch 7/500
394/394 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.1129 - mae: 0.2637 - val_loss: 0.0906 - val_mae: 0.2074 - learning_rate: 0.0010
Epoch 8/

2026-02-28 19:10:40.940138: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-28 19:10:40.940173: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-28 19:10:42.869407: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_47', 16 bytes spill stores, 16 bytes spill loads

2026-02-28 19:10:45.500878: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : R

365/394 ━━━━━━━━━━━━━━━━━━━━ 0s 971us/step

2026-02-28 19:10:46.876923: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-28 19:10:46.876958: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-28 19:10:49.283070: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_47', 16 bytes spill stores, 16 bytes spill loads

2026-02-28 19:10:50.473192: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : R

394/394 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step
    Test Correlation MAE: 27.4847 mHa
    Model saved to deployment_objects/NN_H70_DeltaHF.keras
